# 00 - Foundations and Setup
## Shakespeare Evidence Desk


## Why This Stage

*In general — what foundations are and why any RAG system needs them first.* A retrieval-augmented system is only as trustworthy as the data and infrastructure beneath it. Before a single document is parsed or a single answer generated, you need a corpus you can trust and reproduce, together with the shared services every later stage assumes already exist — model access, telemetry, and a labeled set of questions to measure against. Foundations is the stage that establishes this fixed, known starting point so nothing downstream is built on sand.

It is deliberately *not* RAG itself: no passages are created and no model is asked to answer. Its only job is to take setup and data drift off the list of things that could later explain a result. Once the ground truth is fixed and the plumbing works, every subsequent stage can credibly claim its numbers reflect the pipeline rather than an accident of the environment.


## Conceptual Foundations: What RAG Is and Where This Stage Fits

Because this is the first notebook in the series, it is also the place to establish the vocabulary and the mental model every later chapter assumes. Read this once; later notebooks will reference these terms rather than redefine them.

### What problem does RAG solve?

A large language model (LLM) answers from a frozen snapshot of what it saw during training. That creates three practical problems for a serious application:

- **Staleness** — it knows nothing added after its training cut-off, and nothing private to your organization.
- **Hallucination** — when it does not know, it often *guesses fluently*, producing confident text with no basis in any source.
- **No provenance** — even when it is right, it cannot point to *where* the answer came from, so you cannot verify or cite it.

**Retrieval-Augmented Generation (RAG)** addresses all three by separating *knowledge* from *reasoning*. Instead of trusting the model's memory, we keep the authoritative text in an external store, **retrieve** the few passages relevant to each question at query time, and ask the model to answer **using only those passages**. The model becomes a reader and writer over supplied evidence rather than an oracle recalling from memory.

> **Our running use case — the "Shakespeare Evidence Desk."** Throughout this course the corpus is **ten complete Shakespeare plays** from Project Gutenberg. The system must answer questions like *"By what means does the Ghost say King Hamlet was killed?"* by quoting the actual lines, and must **refuse** questions the ten plays cannot support (e.g. a plot point from a play not in the corpus). Shakespeare is a deliberate choice: it is public-domain (no licensing friction), richly structured (acts, scenes, speakers — ideal for teaching provenance), and familiar enough that wrong answers are easy to spot.

### The complete pipeline (and where each notebook lives)

RAG splits into an **offline/indexing path** (done once, ahead of time) and an **online/query path** (run per question). This series walks the whole thing:

```
 OFFLINE  (build the knowledge store)          ONLINE  (answer a question)
 ────────────────────────────────────          ──────────────────────────────
 00  Foundations & setup      ← you are here
 01  Ingestion & parsing                        05  Retrieval  (search the index)
 02  Chunking                                    06  Reranking  (reorder candidates)
 03  Embeddings                                  07  Context building
 04  Indexing                                    08  Grounded generation
                                                 09  Query handling / adaptive RAG
 10  Continuous evaluation   (judges the whole system, offline + online)
 11  Production & serving     12  Operations & safety
```

| Term | Plain meaning | Introduced in |
|---|---|---|
| **Corpus** | the raw source documents (our ten plays) | 00 |
| **Parsing** | turning raw files into clean, structured units (act/scene/speaker) | 01 |
| **Chunk** | a passage-sized piece of text — the unit that gets embedded and retrieved | 02 |
| **Embedding** | a chunk turned into a vector (a list of numbers) capturing its meaning | 03 |
| **Index** | a data structure over the vectors that makes nearest-neighbor search fast | 04 |
| **Retrieval** | finding the chunks most relevant to a query | 05 |
| **Grounding** | forcing the model to answer only from retrieved evidence | 08 |
| **Provenance** | the trail from an answer back to the exact source lines that support it | all stages |
| **Handoff** | a saved `.artifacts/*.json` file passing one stage's output to the next | all stages |

### How this course is structured (and why it matters pedagogically)

Every stage notebook follows the same disciplined loop: **load** the previous stage's handoff → **explain the alternatives** → **run a controlled experiment** (vary one factor, hold the rest fixed) → **measure** against the labeled questions → **justify** a selection with the numbers → **export** a handoff for the next stage. This mirrors how real RAG systems are tuned — one measured decision at a time — and it is why we build the **labeled evaluation set** (Step 14) and **telemetry** (Step 13) *before* any RAG exists: you cannot improve what you cannot measure.

### What goes wrong if Foundations is skipped or done poorly

- **Unreproducible corpus** → later "improvements" may just be different input text, not better engineering.
- **No labeled questions** → every downstream claim ("better retrieval") is an opinion, not a measurement.
- **No telemetry** → when an answer is wrong in notebook 08, you have no trace to explain *why*.
- **Secrets in code** → keys committed to Git are the single most common RAG security failure; this stage establishes keyless identity auth instead.

Foundations is not glamorous, but it is the stage that makes every later number trustworthy.


- **Goal:** prepare a reliable collection of ten Shakespeare plays in Azure so later notebooks can focus on learning RAG.
- **Not RAG yet:** this notebook does not parse the plays into passages, create embeddings, retrieve answers, or call an LLM.
- **Where it runs:** Python and the notebook kernel run on your machine. Azure stores the documents. Python invokes Azure CLI using your existing login.
- **Reading pattern:** read the Markdown explanation, run the following code cell, and check its result before moving on. Later cells use values created by earlier ones.
- **Repeatability:** reuse the existing setup by default. Four explicit switches control optional cloud changes.
- **Privacy:** subscription/resource/user identifiers come from `.env`. Generated setup files go in the ignored `.artifacts/` directory. Clear identifying outputs before sharing.

### Terms We Use

- **Subscription:** the Azure account scope under which resources are managed and billed.
- **Resource group:** a named group that organizes related Azure resources.
- **Storage account:** the Azure resource providing blob storage.
- **Container:** a collection of blobs inside the storage account. A blob is a stored file/object.
- **Corpus:** our collection of ten complete plays.
- **Manifest:** a JSON inventory of those files, their source URLs, sizes, and checksums.
- **Snapshot:** our name for a dated folder containing one corpus download; not Azure's separate blob-snapshot feature.
- **Handoff:** a saved setup report that lets the next notebook locate the verified corpus without reusing this kernel's variables.


## 0. One-Time Environment Setup

- **Purpose:** create an isolated Python environment and kernel so this notebook runs reliably. Do this once per machine, before Step 1.
- **Why needed:** the system Python (`/usr/bin/python3`) ships without `pip` or `ensurepip`, and installing into it triggers VS Code's "global environment" warning. A project-local `.venv` avoids version conflicts.

### Steps (run in the integrated terminal from the repository root)

```bash
# 1. Install the venv module for the system Python (needs sudo once).
sudo apt update
sudo apt install -y python3.12-venv

# 2. Create a project-local virtual environment.
python3 -m venv .venv

# 3. Install dependencies into the venv (note: 'pip', not 'pip3').
./.venv/bin/python -m pip install -r requirements.txt
```

### Select the environment in VS Code

- **Interpreter:** open the Command Palette → **Python: Select Interpreter** → choose **`.venv` (`./.venv/bin/python`)**, not the Global `/usr/bin/python3`.
- **Notebook kernel:** click the kernel picker (top right) → **Select Another Kernel → Python Environments → `.venv`**. Confirm the path ends in `.venv/bin/python`.
- **If `.venv` is not listed:** run **Developer: Reload Window**, then reselect it.

### Terminal `.env` variables (optional)

- VS Code may warn that "terminal environment injection is disabled." This repo enables it via `.vscode/settings.json` (`"python.terminal.useEnvFile": true`). Open a **new** terminal after the first run so the setting takes effect.

### Verify

```bash
./.venv/bin/python -c "import pip, ipykernel; print('pip', pip.__version__, '| ipykernel', ipykernel.__version__)"
```

- **Expected result:** prints the installed `pip` and `ipykernel` versions, confirming the kernel can run the cells below.


## 1. Dependencies and Local Setup

- **Purpose:** prepare the local Python tools and file locations used by later cells. Nothing is created in Azure in this step.
- **Before running:** select the project `.venv` kernel and install dependencies in it with `python -m pip install -r requirements.txt`.
- **External commands:** Azure CLI (`az`) communicates with Azure; `curl` downloads public texts; `jq` builds JSON metadata; `shasum` calculates file fingerprints. The uploader uses the last three commands.
- **Imports:** the next cell loads Python utilities for JSON, paths, subprocess commands, temporary files, dates, checksums, and reading `.env`.
- **`ROOT`:** finds the repository folder, using `.env.example` as a marker. Starting in the wrong folder raises an error instead of loading the wrong configuration.
- **`ENV_FILE`:** points to the private `.env` file. If it is missing, copy `.env.example` to `.env` and fill in your own settings before Step 2.
- **`ARTIFACTS`:** creates the local `.artifacts/` folder if needed. This holds generated reports, not your Azure-hosted corpus.
- **Authentication prerequisite:** run interactive `az login` in your terminal for the appropriate tenant. Reading `.env` does not log you in.
- **Expected result:** 'Repository and local artifact directory ready.' No storage account, role, container, or corpus is changed.

In [ ]:
import hashlib
import json
import re
import shutil
import subprocess
import tempfile
from datetime import datetime, timezone
from pathlib import Path
from uuid import UUID

from dotenv import dotenv_values, set_key

ROOT = Path.cwd().resolve()
if not (ROOT / '.env.example').is_file() and (ROOT.parent / '.env.example').is_file():
    ROOT = ROOT.parent
if not (ROOT / '.env.example').is_file():
    raise RuntimeError('Use the repository as the notebook working directory.')
ENV_FILE = ROOT / '.env'
ARTIFACTS = ROOT / '.artifacts'
ARTIFACTS.mkdir(exist_ok=True)
print('Repository and local artifact directory ready.')

## 2. Load and Validate Private Configuration

- **Purpose:** tell the notebook which Azure destination and user to use without hard-coding identifiers into code.
- **Input:** the private `.env` file. The checked-in `.env.example` is a template, not working configuration.
- **What the code does:** reads non-empty values into the Python `config` dictionary, checks required keys, and validates identifier/name formats before making Azure calls.
- **What it does not prove:** valid-looking names and IDs do not guarantee resources exist or that you have permission. Later steps check those against Azure.

### Configuration Values

- **`AZURE_SUBSCRIPTION_ID`:** chooses the subscription for every CLI call. The notebook does not change the global CLI default.
- **`AZURE_RESOURCE_GROUP`:** names the group containing the storage account.
- **`AZURE_LOCATION`:** chooses the region when creating missing resources; it does not move existing ones.
- **`AZURE_STORAGE_ACCOUNT`:** names the storage resource. Azure requires a globally unique name using 3-24 lowercase letters/digits.
- **`AZURE_STORAGE_CONTAINER`:** names the collection of blobs inside that account.
- **`AZURE_PRINCIPAL_ID`:** identifies the user to inspect or grant access to. It is your user object ID in the target tenant, not your email or subscription ID.
- **`AZURE_CORPUS_PREFIX`:** selects the dated corpus folder to reuse. It may be empty before the first upload, but Step 8 requires a valid selection or a new upload.
- **`AZURE_SETUP_HANDOFF_BLOB`:** optional reference written only when the setup report is uploaded in Step 10.

- **Expected result:** required configuration is validated without printing its values. If a key is missing or malformed, fix `.env` and rerun this cell.
- **Privacy:** keep `.env` private, ignored by Git, and shell-compatible. The uploader sources this trusted local file; never paste its contents into shared notebook outputs.

In [ ]:
if not ENV_FILE.is_file():
    raise RuntimeError('Create .env from .env.example and configure it first.')
config = {key: value for key, value in dotenv_values(ENV_FILE).items() if value}
required_keys = (
    'AZURE_SUBSCRIPTION_ID', 'AZURE_RESOURCE_GROUP', 'AZURE_LOCATION',
    'AZURE_STORAGE_ACCOUNT', 'AZURE_STORAGE_CONTAINER', 'AZURE_PRINCIPAL_ID',
)
missing_keys = [key for key in required_keys if not config.get(key)]
if missing_keys:
    raise RuntimeError('Missing configuration keys: ' + ', '.join(missing_keys))
UUID(config['AZURE_SUBSCRIPTION_ID'])
UUID(config['AZURE_PRINCIPAL_ID'])
if not re.fullmatch(r'[a-z0-9]{3,24}', config['AZURE_STORAGE_ACCOUNT']):
    raise ValueError('Storage account names require 3-24 lowercase letters or digits.')
container_name = config['AZURE_STORAGE_CONTAINER']
if not re.fullmatch(r'[a-z0-9][a-z0-9-]{1,61}[a-z0-9]', container_name) or '--' in container_name:
    raise ValueError('Use a valid lowercase container name without consecutive hyphens.')
print('Required configuration validated; identifiers are not displayed.')

## 3. Switches: Decide What This Run Is Allowed to Change

- **What is a switch?** A Python variable set to `True` or `False`. Later cells read it to decide whether an optional action is allowed. Setting it does not perform the action immediately.
- **`False` means:** do not perform that optional action. Checks and other enabled actions still run.
- **`True` means:** allow the relevant later cell to perform the action. It does not bypass Azure permissions or guarantee success.
- **Where to change them:** edit the four assignments in the next code cell, run that cell, then run the relevant later sections. Editing a value without rerunning the cell does not update the kernel.

### `CREATE_INFRASTRUCTURE` — Used in Steps 5 and 7

- With `False`, reuse the configured resource group, storage account, and container. Stop with an error if one is missing.
- With `True`, create only missing resources. Existing resources are reused; their settings are not silently changed.
- Enable this for a new Azure destination after reviewing names, region, permissions, and possible charges.

### `GRANT_DATA_ACCESS` — Used in Step 6

- With `False`, inspect assigned roles without adding a role.
- With `True`, grant Storage Blob Data Contributor at storage-account scope if no direct/inherited write-capable data role is found.
- The configured principal must match the signed-in user. Your identity must also have role-assignment authority; the switch cannot grant itself that authority.
- Enable this to repair missing blob access, not on every run. New permissions can take minutes to propagate.

### `UPLOAD_NEW_CORPUS` — Used in Step 8

- With `False`, reuse the snapshot selected by `AZURE_CORPUS_PREFIX` in `.env`.
- With `True`, download and upload another ten-play collection into a new dated folder, then update the selected prefix in `.env`.
- It does not overwrite or delete previous collections. Repeated enabled runs create additional copies and can incur charges.

### `UPLOAD_SETUP_HANDOFF` — Used in Step 10

- With `False`, save the setup report only in the local ignored artifact directory.
- With `True`, also upload that report to Azure and save its blob reference in `.env`.
- This uploads a small report, not another copy of the plays. The local report is sufficient for the next notebook on this machine.

### Recommended Settings

- **Your existing setup:** leave all four switches `False`; the storage, permission, and corpus are already configured.
- **Fresh setup:** enable `CREATE_INFRASTRUCTURE` and `UPLOAD_NEW_CORPUS`; enable `GRANT_DATA_ACCESS` only if a grant is needed and you are authorized. Keep Azure handoff upload optional.
- **After setup succeeds:** return the switches to `False` and rerun the switch cell before the next verification run.
- **Even with all switches `False`:** Azure read operations and sample downloads still happen, and local artifact files are written. This is read-only for cloud resources, not a no-op or a completely offline run.

In [ ]:
CREATE_INFRASTRUCTURE = False
GRANT_DATA_ACCESS = False
UPLOAD_NEW_CORPUS = False
UPLOAD_SETUP_HANDOFF = False
print('Execution switches configured. Review them before continuing.')

## 4. Azure CLI Helper and Login Check

- **Purpose:** provide one small function for later Azure commands and confirm your CLI login can access the selected subscription.
- **Dependency check:** `shutil.which('az')` checks whether Azure CLI is available on your machine. If not, install it before continuing.
- **`az_json(...)`:** takes command arguments, adds the configured subscription and JSON output options, and runs Azure CLI as a local subprocess.
- **Why JSON:** Azure's response becomes a Python dictionary, list, or other JSON value that later code can inspect instead of reading formatted terminal tables.
- **Errors:** a failing command stops execution with its diagnostic. Do not continue using incomplete results, and clear diagnostics containing resource identifiers before sharing.
- **Login check:** `account show` verifies that the selected subscription resolves correctly. This confirms access to the subscription, not yet to blob contents.
- **Authentication:** the function uses your existing Azure CLI login. It does not call `az login`, fetch account keys, or print access tokens.
- **Expected result:** 'Azure CLI can access the configured subscription.' Defining the helper itself creates no cloud resources.

In [ ]:
if shutil.which('az') is None:
    raise RuntimeError('Install Azure CLI before continuing.')

def az_json(*arguments):
    command = [
        'az', *arguments, '--subscription', config['AZURE_SUBSCRIPTION_ID'],
        '--only-show-errors', '--output', 'json',
    ]
    result = subprocess.run(command, capture_output=True, text=True, timeout=300)
    if result.returncode != 0:
        raise RuntimeError(
            'Azure CLI failed. Check login, permissions, and role propagation. '
            'Clear diagnostic outputs before sharing.\n' + result.stderr.strip()
        )
    return json.loads(result.stdout) if result.stdout.strip() else None

subscription = az_json('account', 'show')
if subscription['id'].lower() != config['AZURE_SUBSCRIPTION_ID'].lower():
    raise RuntimeError('CLI did not resolve the configured subscription.')
print('Azure CLI can access the configured subscription.')

## 5. Create or Reuse the Resource Group and Storage Account

- **Purpose:** ensure the Azure storage destination exists and follows our initial security choices. This does not upload the plays or create the container yet.
- **First check:** ask Azure whether the configured resource group exists. It organizes the storage account and other related resources.
- **Second check:** look for the configured storage account inside that group.
- **With `CREATE_INFRASTRUCTURE = False`:** reuse existing resources. If the group or account is missing, stop rather than create something unexpectedly.
- **With `CREATE_INFRASTRUCTURE = True`:** create only missing resources using the names and region from `.env`. Existing account settings are inspected, not silently rewritten.

### Initial Storage Choices

- **StorageV2 / Standard:** a general-purpose storage account suitable for storing our text files and generated artifacts.
- **LRS:** keeps redundant copies within one location. It is a cost-conscious learning choice, not protection against a regional disaster.
- **Hot access:** suits files we expect to read during repeated experiments. Production requirements may justify different settings.
- **HTTPS and minimum TLS 1.2:** require encrypted transport to the storage service.
- **Anonymous blob access disabled:** unauthenticated visitors cannot read the files.
- **Shared keys disabled:** blob access uses Microsoft Entra identity rather than storage-account keys.

- **Values saved for later:** `storage_scope` identifies the account for a narrowly scoped role grant; `blob_endpoint` is the service address used in the setup report.
- **Expected result:** the account exists and its checked security settings match the baseline. If an existing account differs, the code asks you to review it instead of changing it automatically.

In [ ]:
resource_group = config['AZURE_RESOURCE_GROUP']
group_exists = az_json('group', 'exists', '--name', resource_group)
if not group_exists:
    if not CREATE_INFRASTRUCTURE:
        raise RuntimeError('Group missing. Enable CREATE_INFRASTRUCTURE if creation is approved.')
    az_json('group', 'create', '--name', resource_group, '--location', config['AZURE_LOCATION'])
accounts = az_json('storage', 'account', 'list', '--resource-group', resource_group)
account = next((item for item in accounts if item['name'] == config['AZURE_STORAGE_ACCOUNT']), None)
if account is None:
    if not CREATE_INFRASTRUCTURE:
        raise RuntimeError('Account missing. Enable CREATE_INFRASTRUCTURE if creation is approved.')
    account = az_json(
        'storage', 'account', 'create', '--resource-group', resource_group,
        '--name', config['AZURE_STORAGE_ACCOUNT'], '--location', config['AZURE_LOCATION'],
        '--kind', 'StorageV2', '--sku', 'Standard_LRS', '--access-tier', 'Hot',
        '--https-only', 'true', '--min-tls-version', 'TLS1_2',
        '--allow-blob-public-access', 'false', '--allow-shared-key-access', 'false',
    )
account = az_json(
    'storage', 'account', 'show', '--resource-group', resource_group,
    '--name', config['AZURE_STORAGE_ACCOUNT'],
)
if account.get('allowBlobPublicAccess') is not False or account.get('allowSharedKeyAccess') is not False:
    raise RuntimeError('Review the existing account: expected anonymous access and shared keys to be disabled.')
if account.get('enableHttpsTrafficOnly') is not True or account.get('minimumTlsVersion') not in ('TLS1_2', 'TLS1_3'):
    raise RuntimeError('Review the existing account HTTPS and minimum TLS settings.')
storage_scope = account['id']
blob_endpoint = account['primaryEndpoints']['blob'].rstrip('/')
print('Storage account verified: anonymous access and shared keys are disabled.')

## 6. Diagnose and Optionally Fix Blob Permissions

- **Purpose:** address the portal error saying your Microsoft Entra user cannot list blob data.
- **Key distinction:** managing a storage account and accessing its files are different permissions. Contributor can manage the resource but does not automatically grant blob read/write access.
- **Inspect first:** list the configured user's direct and inherited role assignments at the storage-account scope. Look for Storage Blob Data Contributor or Storage Blob Data Owner.
- **With `GRANT_DATA_ACCESS = False`:** report what was found without adding a role. This is the setting to use when access is already configured.
- **With `GRANT_DATA_ACCESS = True`:** if no listed write-capable data role is found, check that the configured principal matches your signed-in user, then request Storage Blob Data Contributor.
- **What that role allows:** reading and uploading blob data, along with other blob data operations. The grant is limited to this storage account, not the entire subscription.
- **Required authority:** you must already be authorized to assign roles, for example through User Access Administrator or Owner. The switch cannot manufacture this authority.
- **Important limits:** group-based access, custom roles, and privileged-role activation can affect effective permissions. The role list is a diagnostic; Step 7 tests actual access.
- **Expected result:** an existing role is reported, a missing role is granted when enabled and authorized, or the notebook explains that access still needs attention.
- **After a new grant:** allow minutes for propagation and refresh the portal using the same tenant/user. Do not enable keys or anonymous access to work around the error.

In [ ]:
assignments = az_json(
    'role', 'assignment', 'list', '--assignee', config['AZURE_PRINCIPAL_ID'],
    '--scope', storage_scope, '--include-inherited',
)
write_roles = {'Storage Blob Data Contributor', 'Storage Blob Data Owner'}
has_write_role = any(item.get('roleDefinitionName') in write_roles for item in assignments)
if GRANT_DATA_ACCESS and not has_write_role:
    signed_in_user = az_json('ad', 'signed-in-user', 'show', '--query', 'id')
    if signed_in_user.lower() != config['AZURE_PRINCIPAL_ID'].lower():
        raise RuntimeError('Configured principal differs from the signed-in user; review .env before granting access.')
    az_json(
        'role', 'assignment', 'create', '--assignee-object-id', config['AZURE_PRINCIPAL_ID'],
        '--assignee-principal-type', 'User', '--role', 'Storage Blob Data Contributor',
        '--scope', storage_scope,
    )
    print('Data role granted at account scope. Allow propagation before testing access.')
elif has_write_role:
    print('A direct/inherited write-capable blob data role is already assigned.')
else:
    print('No direct/inherited write role found. Test access next; request help from an authorized role administrator if needed.')

## 7. Verify Access and Create a Private Container

- **Purpose:** confirm that Azure actually lets your signed-in identity access blob storage, then prepare the collection that will hold the plays.
- **Container meaning:** a named collection of blobs inside the storage account. Each uploaded play becomes a blob inside this collection.
- **Authentication:** every storage command uses `--auth-mode login`, meaning your Azure CLI identity rather than an account key.
- **Existence check:** ask whether the configured container already exists. This is a real data-access operation, unlike just inspecting the storage account's management settings.
- **With `CREATE_INFRASTRUCTURE = False`:** reuse an existing container; stop if it is missing.
- **With `CREATE_INFRASTRUCTURE = True`:** create a missing container with anonymous access disabled. This still requires effective data permissions.
- **Privacy check:** inspect the resulting container and reject an anonymous-access setting.
- **Network distinction:** private here means no anonymous access. It does not mean we created a private network endpoint; authenticated access to the public service endpoint is used initially.
- **Expected result:** data access succeeds and the configured container is private. No plays are uploaded in this step.
- **If it fails after Step 6:** allow role propagation, verify the signed-in tenant/user, and rerun this check. A failure should not trigger a less secure workaround.

In [ ]:
container_exists = az_json(
    'storage', 'container', 'exists', '--account-name', config['AZURE_STORAGE_ACCOUNT'],
    '--name', config['AZURE_STORAGE_CONTAINER'], '--auth-mode', 'login',
)['exists']
if not container_exists:
    if not CREATE_INFRASTRUCTURE:
        raise RuntimeError('Container missing. Enable CREATE_INFRASTRUCTURE if creation is approved.')
    az_json(
        'storage', 'container', 'create', '--account-name', config['AZURE_STORAGE_ACCOUNT'],
        '--name', config['AZURE_STORAGE_CONTAINER'], '--auth-mode', 'login', '--public-access', 'off',
    )
container_properties = az_json(
    'storage', 'container', 'show', '--account-name', config['AZURE_STORAGE_ACCOUNT'],
    '--name', config['AZURE_STORAGE_CONTAINER'], '--auth-mode', 'login',
)
if container_properties.get('properties', {}).get('publicAccess') not in (None, 'None'):
    raise RuntimeError('Container is not private; review anonymous-access settings.')
print('Blob data access succeeded; the configured container is private.')

## 8. Upload the Corpus or Reuse a Snapshot

- **Purpose:** select the ten-play document collection that later RAG stages will use. The corpus is the documents themselves, not embeddings or an index.
- **With `UPLOAD_NEW_CORPUS = False`:** do not download or upload another collection. Reuse the dated folder named by `AZURE_CORPUS_PREFIX` in `.env`. Step 9 checks its contents.
- **With `UPLOAD_NEW_CORPUS = True`:** run `scripts/upload_shakespeare.sh`, which reads the same `.env` and downloads ten complete Shakespeare plays from Project Gutenberg.
- **Before upload:** the script checks each title/author and preserves the original file, including its Gutenberg notices.
- **Metadata:** the script creates a manifest containing source URLs, titles, file sizes, and SHA-256 checksums. This records where each document came from and how to verify it.
- **Storage layout:** upload ten text files and one manifest into a new dated folder. Earlier folders are not overwritten or deleted; repeated new uploads create additional copies.
- **Verification inside the script:** check that eleven blobs exist and that downloading Hamlet returns identical bytes.
- **Selection update:** read the script's local JSON result and update only `AZURE_CORPUS_PREFIX` in `.env`, so future runs use the new collection.
- **Expected result:** a corpus folder is selected. With uploading disabled, an empty or placeholder prefix raises an error instead of guessing a collection.
- **Rights and later processing:** original Shakespeare works are public domain; verify edition terms and applicable local law. Later parsing excludes administrative notices from searchable content while preserving unchanged originals.
- **Current limitation:** these editions are text-only. Scanned or illustrated editions are a separate future experiment, not something this upload invents.

In [ ]:
if UPLOAD_NEW_CORPUS:
    result = subprocess.run(
        ['bash', str(ROOT / 'scripts' / 'upload_shakespeare.sh')],
        cwd=ROOT, capture_output=True, text=True, timeout=900,
    )
    if result.returncode != 0:
        raise RuntimeError('Corpus upload failed.\n' + result.stderr.strip())
    upload_result = json.loads((ARTIFACTS / 'corpus-upload.json').read_text(encoding='utf-8'))
    if upload_result['document_count'] != 10:
        raise RuntimeError('Uploader did not produce the expected ten-document result.')
    config['AZURE_CORPUS_PREFIX'] = upload_result['prefix']
    set_key(str(ENV_FILE), 'AZURE_CORPUS_PREFIX', upload_result['prefix'], quote_mode='always')
    print('New snapshot uploaded and selected in .env.')
corpus_prefix = config.get('AZURE_CORPUS_PREFIX', '').strip('/')
if not corpus_prefix or corpus_prefix.endswith('your-snapshot'):
    raise RuntimeError('Configure AZURE_CORPUS_PREFIX or enable UPLOAD_NEW_CORPUS.')
print('Snapshot selected; its location remains in private configuration.')

## 9. Validate Inventory and a Download Checksum

- **Purpose:** prove that the selected corpus is present and consistent before a later stage processes it. A successful upload message alone is not enough.
- **Read the manifest:** download its JSON into a temporary local folder and read the ten document entries.
- **Check document identities:** require the expected ten distinct Gutenberg works and the author William Shakespeare.
- **Check metadata:** ensure each document refers to the selected folder, has a positive byte count, and has a correctly formatted SHA-256 checksum.
- **Compare with Azure:** list blobs in that folder. Their names must match the ten texts plus the manifest, and each text's size must match the inventory.
- **What is a checksum?** A fingerprint calculated from file contents. Different bytes are overwhelmingly likely to produce a different SHA-256 fingerprint.
- **Verify Hamlet:** download its actual contents, calculate SHA-256 locally, and compare that value with the manifest. This checks Hamlet's full contents; it does not fully hash all ten uploaded files.
- **Local cleanup:** verification downloads use a temporary directory that is removed when the check completes.
- **Expected result:** ten titles/sizes and a confirmation that ten plays, one manifest, and Hamlet's checksum were verified. Azure identifiers are not intentionally printed.
- **What this does not evaluate:** literary answers, parsing accuracy, retrieval quality, or model performance. Those require later experiments.

In [ ]:
def download_blob(blob_name, file_path):
    return az_json(
        'storage', 'blob', 'download', '--account-name', config['AZURE_STORAGE_ACCOUNT'],
        '--container-name', config['AZURE_STORAGE_CONTAINER'], '--name', blob_name,
        '--file', str(file_path), '--auth-mode', 'login',
    )

manifest_blob = corpus_prefix + '/manifest.json'
with tempfile.TemporaryDirectory() as directory:
    temporary_path = Path(directory)
    manifest_path = temporary_path / 'manifest.json'
    download_blob(manifest_blob, manifest_path)
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    documents = manifest['documents']
    expected_ids = {f'gutenberg-{ebook_id}' for ebook_id in (1524, 1533, 1513, 1531, 1532, 1514, 1515, 1522, 1540, 1526)}
    if len(documents) != 10 or {document['id'] for document in documents} != expected_ids:
        raise RuntimeError('Manifest does not contain the expected ten distinct plays.')
    for document in documents:
        if document['author'] != 'William Shakespeare' or not document['original_notices_preserved']:
            raise RuntimeError('Unexpected author or missing original-notices flag.')
        if not document['blob_name'].startswith(corpus_prefix + '/'):
            raise RuntimeError('Manifest points outside the selected snapshot.')
        if document['bytes'] <= 0 or not re.fullmatch(r'[0-9a-f]{64}', document['sha256']):
            raise RuntimeError('Invalid document size or checksum metadata.')
    blobs = az_json(
        'storage', 'blob', 'list', '--account-name', config['AZURE_STORAGE_ACCOUNT'],
        '--container-name', config['AZURE_STORAGE_CONTAINER'], '--prefix', corpus_prefix + '/',
        '--auth-mode', 'login',
    )
    blob_sizes = {blob['name']: blob['properties']['contentLength'] for blob in blobs}
    expected_names = {document['blob_name'] for document in documents} | {manifest_blob}
    if set(blob_sizes) != expected_names:
        raise RuntimeError('Inventory does not match the ten texts and manifest.')
    if any(blob_sizes[document['blob_name']] != document['bytes'] for document in documents):
        raise RuntimeError('Document sizes do not match the manifest.')
    hamlet = next(document for document in documents if document['id'] == 'gutenberg-1524')
    hamlet_path = temporary_path / 'hamlet.txt'
    download_blob(hamlet['blob_name'], hamlet_path)
    if hashlib.sha256(hamlet_path.read_bytes()).hexdigest() != hamlet['sha256']:
        raise RuntimeError('Hamlet checksum verification failed.')
for document in documents:
    print(f"{document['title']}: {document['bytes']:,} bytes")
print('Verified ten plays, one manifest, and the Hamlet checksum.')

## 10. Save a Setup Report for the Next Notebook

- **Purpose:** save the information the next notebook needs to locate our verified documents. A handoff is just this saved setup report, not a new Azure service.
- **Why save it?** Python variables disappear when the kernel restarts. A JSON file survives and can be read independently by the ingestion notebook.
- **This is not another corpus copy:** the report contains references and metadata, not the complete text of the plays. It does not parse, embed, or index documents, and does not call an LLM.

### What the `handoff` Dictionary Contains

- **`stage` and `created_at`:** identify which setup step produced the report and when it ran.
- **`storage`:** records the selected subscription, resource group, storage account, container, and blob service address. Values come from private configuration and verified Azure results.
- **`corpus_prefix` and `source_manifest_blob`:** identify the exact corpus folder and its manifest, rather than asking the next notebook to guess the newest folder.
- **`documents`:** copies the manifest's file references, source information, sizes, and checksums into the report.
- **`verification`:** records the checks completed in Step 9: ten documents, eleven blobs, and the verified Hamlet checksum. It is a setup result, not a RAG quality score.
- **`decision`:** explains the initial choices and the next experiment. Ten related plays support cross-work questions; identity authentication avoids shared keys; Standard LRS is an initial learning choice to revisit for production resilience.

### What Gets Written

- **Always, after earlier steps succeed:** save `.artifacts/setup-handoff.json` locally. Rerunning this step replaces that local report with the current verified setup.
- **How the save works:** `json.dumps(handoff, indent=2)` converts the Python dictionary into readable JSON text; `write_text(...)` writes that text into the file.
- **With `UPLOAD_SETUP_HANDOFF = False`:** stop at the local report. Nothing is uploaded by this step, and `.env` is not changed.
- **With `UPLOAD_SETUP_HANDOFF = True`:** also upload the report to a new Azure blob and record its reference as `AZURE_SETUP_HANDOFF_BLOB` in `.env`. It does not upload the plays again.
- **Recommended now:** leave the upload switch `False`. The next notebook on this machine can read the local report.
- **Expected result:** 'Local handoff saved in .artifacts/setup-handoff.json.' We can now restart the kernel without losing the information needed to find the corpus.
- **Privacy:** the report contains your Azure identifiers and is ignored by Git. Do not publish it or identifying notebook outputs; a cloud copy remains in the private container.

In [ ]:
handoff = {
    'stage': '00-foundations-storage',
    'created_at': datetime.now(timezone.utc).isoformat(),
    'storage': {
        'subscription_id': config['AZURE_SUBSCRIPTION_ID'],
        'resource_group': config['AZURE_RESOURCE_GROUP'],
        'account': config['AZURE_STORAGE_ACCOUNT'],
        'container': config['AZURE_STORAGE_CONTAINER'],
        'blob_endpoint': blob_endpoint,
    },
    'corpus_prefix': corpus_prefix,
    'source_manifest_blob': manifest_blob,
    'documents': documents,
    'verification': {'document_count': len(documents), 'blob_count': len(blobs), 'hamlet_sha256_verified': True},
    'decision': {
        'corpus': 'Ten related Shakespeare plays support passage and cross-work questions.',
        'authentication': 'Microsoft Entra identity; no account keys or anonymous blob access.',
        'storage': 'Standard LRS is the learning baseline; review production resilience separately.',
        'next_stage': 'Compare deterministic and model-assisted parsing while preserving provenance.',
    },
}
handoff_path = ARTIFACTS / 'setup-handoff.json'
handoff_path.write_text(json.dumps(handoff, indent=2) + '\n', encoding='utf-8')
if UPLOAD_SETUP_HANDOFF:
    run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    handoff_blob = f'setup/{run_id}/handoff.json'
    az_json(
        'storage', 'blob', 'upload', '--account-name', config['AZURE_STORAGE_ACCOUNT'],
        '--container-name', config['AZURE_STORAGE_CONTAINER'], '--name', handoff_blob,
        '--file', str(handoff_path), '--auth-mode', 'login', '--overwrite', 'false',
    )
    set_key(str(ENV_FILE), 'AZURE_SETUP_HANDOFF_BLOB', handoff_blob, quote_mode='always')
    print('Setup handoff uploaded; its reference is stored in .env.')
print('Local handoff saved in .artifacts/setup-handoff.json.')

## 11. Connect Hosted Models

- **Purpose:** confirm the notebook can reach the two Azure OpenAI deployments in Foundry that later stages rely on: one chat model and one embedding model. This step only checks connectivity; it does not build RAG.
- **One resource, two deployments:** both deployments live in the same Azure OpenAI resource, so a single endpoint and one identity-based authentication setup serve chat and embeddings.
- **Authentication:** reuses your `az login` identity through `DefaultAzureCredential` with the Cognitive Services token scope. No API keys are stored in `.env` or printed, matching the keyless pattern used for storage.
- **Prerequisite role:** your user needs **Cognitive Services OpenAI User** on the resource. Without it the calls below fail with an authorization error, not a code error.

### Authentication: Keyless vs. API Key

- **Do we need the portal API key?** No. This notebook uses **Microsoft Entra identity** (keyless), the same approach as storage. The key shown in Foundry is only for the alternative key-based method.
- **Keyless (chosen):** your `az login` identity authenticates each call; nothing secret is stored in `.env`. It requires the **Cognitive Services OpenAI User** role on the resource and is revocable per user. Prefer this because `.env` is shared as `.env.example`.
- **API key (fallback):** paste the resource key into a private `AZURE_OPENAI_API_KEY` entry in `.env` and pass it as `api_key` instead of a token provider. It needs no role assignment but is a long-lived secret that leaks if `.env` is shared or committed. Use it only when you cannot be granted the role.

### Configuration Values (all from `.env`)

- **`AZURE_OPENAI_ENDPOINT`:** the resource endpoint. In Foundry this is the **Project endpoint**, shaped like `https://your-foundry-resource.services.ai.azure.com` (the newer equivalent of `https://your-resource.openai.azure.com/`). Use the base URL without any `/openai/...` suffix; the client appends that path. Ignore the portal's API key — this notebook authenticates with your Entra identity.
- **`AZURE_OPENAI_API_VERSION`:** the REST API version supported by your deployments. Newer models such as `gpt-5.4-mini` require a recent version; `2025-04-01-preview` is a safe default. Bump it if a deployment returns a `404`.
- **`AZURE_OPENAI_CHAT_DEPLOYMENT`:** the deployment name of the chat model (here, `gpt-5.4-mini`). This is the deployment name you chose in Foundry, not the base model name.
- **`AZURE_OPENAI_EMBEDDING_DEPLOYMENT`:** the deployment name of the embedding model (here, `text-embedding-3-small`).

### What the Code Does

- Validates the four model keys are present, then builds one `AzureOpenAI` client authenticated with your Entra identity.
- Sends a minimal chat request and a single embedding request to prove both deployments respond.
- Prints only reply length and embedding dimensions; it does not print identifiers, endpoints, or model output text.

- **Expected result:** 'Chat deployment reachable...' and 'Embedding deployment reachable...' with a vector dimension count (1536 for `text-embedding-3-small`). A `401`/`403` means the role is missing or still propagating; a `404` usually means a deployment name or API version mismatch.


In [ ]:
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import AzureOpenAI

model_keys = (
    'AZURE_OPENAI_ENDPOINT', 'AZURE_OPENAI_API_VERSION',
    'AZURE_OPENAI_CHAT_DEPLOYMENT', 'AZURE_OPENAI_EMBEDDING_DEPLOYMENT',
)
missing_model_keys = [key for key in model_keys if not config.get(key)]
if missing_model_keys:
    raise RuntimeError('Missing model configuration keys: ' + ', '.join(missing_model_keys))

token_provider = get_bearer_token_provider(
    DefaultAzureCredential(), 'https://cognitiveservices.azure.com/.default',
)
openai_client = AzureOpenAI(
    azure_endpoint=config['AZURE_OPENAI_ENDPOINT'],
    api_version=config['AZURE_OPENAI_API_VERSION'],
    azure_ad_token_provider=token_provider,
)

chat_response = openai_client.chat.completions.create(
    model=config['AZURE_OPENAI_CHAT_DEPLOYMENT'],
    messages=[{'role': 'user', 'content': 'Reply with the single word: ready'}],
)
chat_reply = (chat_response.choices[0].message.content or '').strip()

embedding_response = openai_client.embeddings.create(
    model=config['AZURE_OPENAI_EMBEDDING_DEPLOYMENT'],
    input='Shakespeare evidence desk connectivity check.',
)
embedding_dimensions = len(embedding_response.data[0].embedding)

print(f'Chat deployment reachable; reply length {len(chat_reply)} characters.')
print(f'Embedding deployment reachable; vector dimensions {embedding_dimensions}.')


## 12. Why Grounding Matters: RAG vs. Ungrounded Prompting

- **Purpose:** show, before building any pipeline, why retrieval-grounded answers are preferable to asking the model from memory alone. This is a motivation demo, not the RAG system.
- **The experiment:** ask the chat model the same question twice — once with **no source text**, and once with a **hand-pasted passage** from the corpus — then compare the answers.
- **What to look for:**
  - **Ungrounded:** the model answers from training memory. It may paraphrase, blend editions, or invent exact wording and line references that look authoritative but are unverifiable.
  - **Grounded:** the model is instructed to answer only from the supplied passage, quote exact text, and admit when the passage does not contain the answer.
- **Why this is not RAG yet:** we paste the passage by hand. A real RAG system *retrieves* the right passage automatically from the indexed corpus — that is what notebooks 01–04 build.
- **Honest framing:** grounding does not make the model always correct. It makes answers **traceable** to a source and makes **abstention** possible. The passage below is a short public-domain excerpt pasted for illustration; do not treat it as the full text or as canonical line numbering.
- **Expected result:** two printed answers. The grounded answer should quote the passage and stay within it; the ungrounded answer typically cannot cite exact supporting text.


In [ ]:
def ask_chat(question, passage=None):
    if passage:
        system = (
            'You are a Shakespeare evidence assistant. Answer only from the provided passage. '
            'Support your answer by quoting exact words from it. '
            'If the passage does not contain the answer, reply that it is not supported by the passage.'
        )
        user = f'Passage:\n"""\n{passage}\n"""\n\nQuestion: {question}'
    else:
        system = 'You are a Shakespeare evidence assistant. Answer the question as accurately as you can.'
        user = question
    response = openai_client.chat.completions.create(
        model=config['AZURE_OPENAI_CHAT_DEPLOYMENT'],
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': user},
        ],
    )
    return (response.choices[0].message.content or '').strip()

question = "According to the Ghost, by what exact means was King Hamlet killed? Quote the supporting words."

# Short public-domain excerpt (Hamlet, Act 1, Scene 5). Illustrative paste, not the full text.
hamlet_ghost_passage = (
    "GHOST.  Sleeping within my orchard,\n"
    "My custom always of the afternoon,\n"
    "Upon my secure hour thy uncle stole,\n"
    "With juice of cursed hebenon in a vial,\n"
    "And in the porches of my ears did pour\n"
    "The leperous distilment; whose effect\n"
    "Holds such an enmity with blood of man\n"
    "That swift as quicksilver it courses through\n"
    "The natural gates and alleys of the body."
)

ungrounded_answer = ask_chat(question)
grounded_answer = ask_chat(question, passage=hamlet_ghost_passage)

print('=== Ungrounded (no passage supplied) ===')
print(ungrounded_answer)
print('\n=== Grounded (answer constrained to the pasted passage) ===')
print(grounded_answer)


## 13. Telemetry: Structured Logs and Traces

- **Purpose:** add observability now so every later stage emits structured logs and traces with shared request IDs, rather than bolting monitoring on at the end.
- **Two complementary signals:**
  - **Structured logs:** one JSON record per model call (request ID, grounded flag, duration, answer size, status). These are cheap, greppable, and good for operational metrics.
  - **Traces:** span-level detail of each call via **OpenTelemetry + OpenInference**. Traces are for evidence-level debugging — latency breakdowns, token usage, and prompt/response attributes.
- **Headless-friendly:** spans are captured **in memory** on this machine as local proof, so the step works on a remote server with no browser and no extra service. In-memory capture is for this demo; later stages export to a collector.
- **Phoenix is optional and decoupled:** to view traces in the **Arize Phoenix** UI, run Phoenix separately (locally or, in production from notebook 08, on Azure Container Apps) and set `PHOENIX_COLLECTOR_ENDPOINT` in `.env`. The same instrumentation then streams spans there via OTLP — no code change.
- **Auto-instrumentation:** `OpenAIInstrumentor` wraps the `openai` client so calls made in Steps 11–12 (and later) automatically produce spans — no manual span code in business logic.
- **Request IDs:** each call gets a `request_id` that appears in both the log and the wrapping span, so operational metrics and detailed traces can be correlated. Do not use high-cardinality IDs as metric labels.
- **Privacy:** logs record sizes and durations, not identifiers or full model output. Restrict and redact telemetry before any export.
- **Phoenix vs. Prometheus — different tools, not rivals:** they answer different questions, so "which is better" depends on the job.
  - **Phoenix (traces/eval):** keeps per-request detail — the prompt, retrieved passages, token usage, and evaluation scores. It answers *"why did this answer hallucinate, and which passage caused it?"* Prometheus structurally cannot: it aggregates per-request detail away, and request IDs as labels would explode its cardinality.
  - **Prometheus (metrics):** time-series counts, gauges, and histograms. It answers *"is the service up, what is the p95 latency, is the error rate over budget?"* and drives alerting. Phoenix is not an SLO/alerting system.
  - **Resolution:** **OpenTelemetry is the shared layer.** The same spans feed Phoenix for debugging/evaluation and reduce to Prometheus/Azure Monitor metrics for dashboards and alerts — instrument once, fan out. This notebook uses Phoenix for the development loop; operational metrics arrive with production (notebook 08).
- **Expected result:** a structured log line plus a count of captured spans (the wrapping `grounded_ask` span and the auto-generated OpenAI span), correlated by `request_id`.


In [ ]:
import logging
import time
import uuid

from openinference.instrumentation.openai import OpenAIInstrumentor
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

# --- Structured logging -------------------------------------------------------
# One JSON line per model call. These lines are the raw material for operational
# metrics later (counts, error rates, latency percentiles).
logging.basicConfig(level=logging.INFO, format='%(message)s')
telemetry_logger = logging.getLogger('shakespeare.telemetry')

# --- OpenTelemetry tracing pipeline -------------------------------------------
# Mental model of the four pieces below:
#   TracerProvider  -> the factory that creates tracers and owns configuration.
#   Resource        -> static labels describing *who* is emitting (the service).
#   SpanProcessor   -> decides what happens to a span when it finishes.
#   SpanExporter    -> where finished spans are sent.
# A "span" is a timed unit of work (one function/API call) with attributes.

# Exporter: keep finished spans in a Python list in this process. No network,
# no server, no browser - ideal on a headless remote. get_finished_spans()
# lets us inspect them below as proof the pipeline works.
span_exporter = InMemorySpanExporter()

# Provider tagged with a service name so traces are attributable to this app.
tracer_provider = TracerProvider(resource=Resource.create({'service.name': 'shakespeare-foundations'}))

# SimpleSpanProcessor exports each span immediately when it ends (vs. batching).
# Simple is fine for a notebook; production uses a BatchSpanProcessor for throughput.
tracer_provider.add_span_processor(SimpleSpanProcessor(span_exporter))

# Optional second destination: if a Phoenix collector URL is set in .env, also
# ship spans there over OTLP/HTTP so you can view them in the Phoenix UI. The
# business logic does not change - we just add another processor/exporter.
phoenix_endpoint = config.get('PHOENIX_COLLECTOR_ENDPOINT')
if phoenix_endpoint:
    from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
    tracer_provider.add_span_processor(
        SimpleSpanProcessor(OTLPSpanExporter(endpoint=phoenix_endpoint.rstrip('/') + '/v1/traces')),
    )

# Auto-instrumentation: monkey-patches the OpenAI SDK so every chat/embedding
# call automatically creates a span (model, token usage, latency) with no extra
# code at the call sites. skip_dep_check avoids a strict version-pin check.
# uninstrument() first makes the cell re-run safe: without it, a previous run's
# patch stays bound to its old provider and new OpenAI spans go to the wrong place.
instrumentor = OpenAIInstrumentor()
instrumentor.uninstrument()
instrumentor.instrument(tracer_provider=tracer_provider, skip_dep_check=True)

# A tracer is what we use to create our own spans (see traced_ask below).
tracer = tracer_provider.get_tracer('shakespeare.foundations')


def traced_ask(question, passage=None):
    # A correlation id that ties the structured log line to the trace span.
    request_id = uuid.uuid4().hex
    started = time.perf_counter()
    status = 'ok'
    answer = ''
    # Open a parent span for the whole operation. The OpenAI call inside creates
    # its own child span automatically (thanks to the instrumentor), so the trace
    # shows grounded_ask -> ChatCompletion as a parent/child pair.
    with tracer.start_as_current_span('grounded_ask') as span:
        span.set_attribute('request_id', request_id)
        span.set_attribute('grounded', passage is not None)
        try:
            answer = ask_chat(question, passage=passage)
        except Exception as exc:
            # Record the failure type on the span, then re-raise so the error is not hidden.
            status = f'error:{type(exc).__name__}'
            raise
        finally:
            # finally runs on both success and failure, so latency/status are always recorded.
            duration_ms = round((time.perf_counter() - started) * 1000)
            span.set_attribute('duration_ms', duration_ms)
            span.set_attribute('status', status)
            telemetry_logger.info(json.dumps({
                'request_id': request_id,
                'grounded': passage is not None,
                'duration_ms': duration_ms,
                'answer_chars': len(answer),  # size only, not the model's text (privacy)
                'status': status,
            }))
    return request_id, answer

# Make one traced call, then read the captured spans back as proof.
trace_request_id, trace_answer = traced_ask(question, passage=hamlet_ghost_passage)
captured_spans = span_exporter.get_finished_spans()
print(f'Captured {len(captured_spans)} spans: ' + ', '.join(sorted({span.name for span in captured_spans})))
print(f'Correlate the structured log and spans by request_id {trace_request_id}.')
if phoenix_endpoint:
    print(f'Also streamed traces to Phoenix at {phoenix_endpoint}.')
else:
    print('Set PHOENIX_COLLECTOR_ENDPOINT in .env to also stream traces to a Phoenix UI.')


## 14. Labeled Evaluation Questions

- **Purpose:** define a small, labeled question set now so later stages can be *measured*, not just demonstrated. Without labels, "better retrieval" or "better answers" is only an opinion.
- **Two disjoint sets:**
  - **Development set:** used while tuning each stage (parsing, chunking, retrieval, generation). You may look at these answers repeatedly.
  - **Held-out set:** reserved for final checks only. Keeping it out of tuning prevents overfitting to the questions you optimized against.
- **Category coverage (per the project plan):** exact quotations, character/event lookups, paraphrases, cross-play comparisons, ambiguous names, and questions **not** answerable from these ten plays.
- **Label against source passages, not chunk IDs:** expected evidence points to a work and a locatable passage (speaker, scene context), so the labels stay valid no matter how later notebooks chunk the text. We deliberately avoid inventing precise canonical line numbers.
- **Unanswerable items matter:** they test whether the system **abstains** instead of fabricating. An answer to an unanswerable question is a failure, not a success.
- **Why so few now:** foundations establishes the *protocol* and schema. The set grows to at least 50 questions before final evaluation (notebook 09); it is not a competition yet.
- **Output:** saved to the ignored `.artifacts/eval-questions.json` so later notebooks load the same labeled set. Expected result prints the counts per set and per category.


In [ ]:
from collections import Counter

# Expected evidence references a work and a locatable passage, not chunk IDs or invented line numbers.
development_questions = [
    {
        'id': 'dev-001', 'category': 'exact_quote', 'answerable': True,
        'question': "In Hamlet, by what means does the Ghost say King Hamlet was killed?",
        'expected_works': ['Hamlet'],
        'expected_evidence': "Ghost in Act 1, Scene 5: poison ('juice of cursed hebenon') poured into the ears.",
    },
    {
        'id': 'dev-002', 'category': 'character_lookup', 'answerable': True,
        'question': "Who speaks the line beginning 'To be, or not to be'?",
        'expected_works': ['Hamlet'],
        'expected_evidence': "Hamlet's soliloquy in Act 3, Scene 1.",
    },
    {
        'id': 'dev-003', 'category': 'paraphrase', 'answerable': True,
        'question': "What does Polonius advise Laertes about borrowing and lending money?",
        'expected_works': ['Hamlet'],
        'expected_evidence': "Polonius in Act 1, Scene 3: advises being neither a borrower nor a lender.",
    },
    {
        'id': 'dev-004', 'category': 'cross_play', 'answerable': True,
        'question': "Compare how a ghost influences the protagonist in Hamlet and in Macbeth.",
        'expected_works': ['Hamlet', 'Macbeth'],
        'expected_evidence': "Hamlet: the Ghost's demand for revenge (Act 1, Scenes 4-5). "
                             "Macbeth: Banquo's ghost at the banquet (Act 3, Scene 4).",
    },
    {
        'id': 'dev-005', 'category': 'ambiguous_name', 'answerable': True,
        'question': "Which plays in this corpus feature a character named Antonio, and what is his role in each?",
        'expected_works': ['The Merchant of Venice', 'Twelfth Night', 'The Tempest'],
        'expected_evidence': "Antonio the merchant (The Merchant of Venice); Antonio the sea captain (Twelfth Night); "
                             "Antonio the usurping brother (The Tempest).",
    },
    {
        'id': 'dev-006', 'category': 'unanswerable', 'answerable': False,
        'question': "What advice does Prospero give in The Winter's Tale?",
        'expected_works': [],
        'expected_evidence': "Not answerable: The Winter's Tale is not in this ten-play corpus. The system should abstain.",
    },
]

heldout_questions = [
    {
        'id': 'held-001', 'category': 'exact_quote', 'answerable': True,
        'question': "In Macbeth, what does Lady Macbeth repeatedly try to wash from her hands while sleepwalking?",
        'expected_works': ['Macbeth'],
        'expected_evidence': "Lady Macbeth in Act 5, Scene 1: an imagined spot of blood ('Out, damned spot').",
    },
    {
        'id': 'held-002', 'category': 'cross_play', 'answerable': True,
        'question': "Name two plays in this corpus that end with the deaths of a pair of young lovers or spouses.",
        'expected_works': ['Romeo and Juliet', 'Othello'],
        'expected_evidence': "Romeo and Juliet (both lovers die); Othello (Othello kills Desdemona, then himself).",
    },
    {
        'id': 'held-003', 'category': 'unanswerable', 'answerable': False,
        'question': "What is the population of Verona according to Romeo and Juliet?",
        'expected_works': [],
        'expected_evidence': "Not answerable: the play does not state a population figure. The system should abstain.",
    },
]

evaluation_questions = {
    'created_at': datetime.now(timezone.utc).isoformat(),
    'schema_version': 1,
    'labeling_note': 'Expected evidence references original source passages (work, speaker, scene context), '
                     'not mutable chunk IDs or invented canonical line numbers. Keep held-out questions out of tuning.',
    'corpus_works': [document['title'] for document in documents],
    'development': development_questions,
    'heldout': heldout_questions,
}

questions_path = ARTIFACTS / 'eval-questions.json'
questions_path.write_text(json.dumps(evaluation_questions, indent=2) + '\n', encoding='utf-8')

dev_categories = Counter(item['category'] for item in development_questions)
held_categories = Counter(item['category'] for item in heldout_questions)
print(f'Saved {len(development_questions)} development and {len(heldout_questions)} held-out questions '
      f'to .artifacts/eval-questions.json.')
print('Development categories: ' + ', '.join(f'{name}={count}' for name, count in sorted(dev_categories.items())))
print('Held-out categories: ' + ', '.join(f'{name}={count}' for name, count in sorted(held_categories.items())))


## Foundations: Techniques, Tools, and Engineering Decisions

The foundations stage is not a single algorithm but a bundle of infrastructure choices. Below is the broader landscape for each, so you can see what this project picked, what the common alternatives are, and how to decide between them in your own system.

Legend: **[IMPLEMENTED]** used here · **[ALTERNATIVE]** common, not used here · **[ADVANCED]** know it exists.

### 1. Corpus storage and access

| Option | What it is | Trade-offs | Status |
|---|---|---|---|
| Azure Blob Storage + identity auth | object store, accessed via Entra login | scalable, auditable, cloud-native; needs an Azure account | **[IMPLEMENTED]** |
| Local filesystem | files in a repo/folder | zero setup, great for prototypes; no sharing, no access control, risks committing data | **[ALTERNATIVE]** |
| S3 / GCS | other cloud object stores | same idea, different cloud; pick by where your compute lives | **[ALTERNATIVE]** |
| Document DB / data lake | Mongo, Delta Lake, etc. | good when documents have rich structured metadata or versioning needs | **[ADVANCED]** |

**How to choose:** prototypes can start on the local filesystem; the moment more than one person or machine needs the same bytes, move to an object store. Choose the cloud that matches your model/compute provider to avoid egress cost and latency.

### 2. Authentication: keyless identity vs API keys

| Approach | How it authenticates | Security posture | Status |
|---|---|---|---|
| Keyless (Microsoft Entra / `DefaultAzureCredential`) | your logged-in identity issues short-lived tokens | **no long-lived secret to leak**; per-user revocable; needs a role assignment | **[IMPLEMENTED]** |
| API key | a long-lived secret string in `.env` | works anywhere with no role setup; the secret leaks if `.env` is shared or committed | **[ALTERNATIVE]** |
| Managed identity (in-cloud) | the compute resource itself has an identity | ideal for production services; no human in the loop | **[ADVANCED]** (arrives in notebook 11) |

**How to choose — the rule this project follows:** prefer keyless whenever you can get the role assignment, because the biggest real-world RAG security incident is a committed key. Fall back to an API key only when you cannot be granted a role, and never commit it. In production, graduate to a managed identity so no secret exists at all.

### 3. Hosted models (chat + embeddings)

This project uses **Azure OpenAI** (`gpt-5.4-mini` for chat, `text-embedding-3-small` for embeddings). Common alternatives you should know: **OpenAI direct**, **Anthropic** / **Google** (chat), and **Cohere**, **Voyage**, or open-weight encoders like **BGE**/**E5**/**Nomic** run locally via `sentence-transformers` (embeddings — explored conceptually in notebook 03). The deciding factors are data-residency and compliance (hosted-in-your-cloud vs third-party API), cost per token, latency, and whether you need on-prem/offline operation (which pushes you toward open-weight local models).

### 4. Observability: three signals, one backbone

| Tool / layer | Answers the question | Role here |
|---|---|---|
| **Structured logs** (JSON lines) | "what happened, how often, how fast?" | **[IMPLEMENTED]** — one record per model call |
| **Traces** (OpenTelemetry + OpenInference) | "*why* did this specific answer go wrong, step by step?" | **[IMPLEMENTED]** — in-memory here, exportable later |
| **Arize Phoenix** | LLM-specific trace viewer + evaluation UI | **[ALTERNATIVE/optional]** — attach via `PHOENIX_COLLECTOR_ENDPOINT` |
| **Prometheus / Azure Monitor** | SLOs, alerting, p95 latency dashboards | **[ADVANCED]** — production (notebook 11) |

**The key insight (don't memorize tools, understand layers):** OpenTelemetry is the shared instrumentation backbone — you instrument once, then fan the same spans out to Phoenix (for debugging/evaluation) *and* to Prometheus/Azure Monitor (for metrics/alerting). Phoenix and Prometheus are not rivals; they answer different questions (per-request forensics vs aggregate health).

### 5. Evaluation-set design (the most under-appreciated choice)

The labeled questions in Step 14 encode several deliberate decisions worth copying:

- **Development vs held-out split** — you tune against development questions and touch the held-out set only at the end, so you don't *overfit to your own test*.
- **Label to source passages, not chunk IDs** — evidence points to "Hamlet, Act 1, Scene 5," which stays valid no matter how notebook 02 later chunks the text.
- **Include unanswerable questions** — these test **abstention**; a system that answers a question the corpus can't support has failed, even if the prose sounds good.
- **Category coverage** — exact quotes, lookups, paraphrases, cross-play comparison, ambiguous names, unanswerable. Diverse categories stop you from optimizing for one easy question type.

**How to choose a starting size:** a handful of carefully labeled questions (we start with 9) is enough to *pre-screen* engineering choices; grow toward 50+ before trusting final numbers. Quality and category coverage matter far more than raw count early on.


## Completion Check and Next Sections

- **For your current setup:** keep all four switches `False`. Restart the kernel and run the code cells in order to confirm the notebook does not depend on hidden state.
- **Successful setup means:** subscription access works, the account passes security checks, the container is private and accessible, ten plays and their manifest are present, Hamlet's checksum matches, and the local handoff exists.
- **After granting a role:** allow propagation and rerun the data-access check. Do not upload again just to fix an authorization error.
- **Before sharing:** clear notebook outputs. `.env`, `.venv/`, and `.artifacts/` are ignored. Share the placeholder `.env.example`, not your configured file or identifying generated reports.
- **What we have achieved:** a known, accessible, verified document collection, two connected hosted models, working telemetry, and a labeled evaluation question set. No embeddings, search index, or RAG answers have been created yet.
- **Foundations now complete:** storage and corpus setup, hosted chat/embedding model connections (Step 11), the grounding motivation demo (Step 12), structured logs and traces (Step 13), and labeled development/held-out questions (Step 14).
- **Next stage notebook:** load the setup report and labeled questions, compare parsing techniques on the selected plays, inspect metrics and errors, justify a choice, and save its outputs for chunking.

## Questions, Discussions, and Further Insights

Deeper discussion points for this stage. The main notebook stands on its own; these are for readers who want to probe the reasoning and common misconceptions.

**Q1. Why build telemetry and an evaluation set *before* there is any RAG to observe or evaluate?**
Because retrofitting them is expensive and biased. If you add logging only after something breaks, you have no trace of the failure that prompted you. If you write test questions after seeing your system's answers, you unconsciously write questions it already passes. Establishing both up front makes every later stage measurable and debuggable from its first run.

**Q2. Is "grounding" the same as "the model is now correct"?**
No — this is the most common misconception about RAG. Grounding (Step 12) makes answers **traceable** to a source and makes **abstention** possible; it does not guarantee correctness. A grounded model can still misread its passage, and retrieval can hand it the wrong passage. Grounding converts "confidently wrong with no evidence" into "answer plus the evidence you can check" — a huge improvement, but not infallibility. The end-to-end quality is what notebooks 08 and 10 actually measure.

**Q3. Why ten related plays instead of one play, or a thousand mixed documents?**
It is a teaching trade-off. One play is too small to show retrieval mattering; thousands of documents add scale and cost that obscure the concepts. Ten related plays are enough to make *cross-document* questions meaningful (compare ghosts in Hamlet and Macbeth), to create genuine ambiguity (three different characters named Antonio), and to keep every stage cheap enough to run in a notebook.

**Q4. The corpus is tiny — wouldn't it be simpler to just paste all ten plays into the prompt?**
For ten plays, long-context prompting ("stuff everything in") can actually work, and it is a legitimate baseline. RAG earns its complexity as the corpus grows past the context window, when cost scales with tokens-per-call, and when you need **provenance** (which exact lines support this answer?) and **freshness** (update one document without re-sending everything). The course uses a small corpus to *teach* RAG mechanics that pay off at scale; Step 12 deliberately contrasts grounded vs ungrounded prompting to make this motivation concrete.

**Q5. What are the main failure modes of this stage specifically?**
(a) **Corpus drift** — silently pointing at a different snapshot, so results aren't reproducible; mitigated by the manifest + checksum verification in Step 9. (b) **Permission confusion** — "I'm a Contributor but can't read blobs," because managing a resource and accessing its data are different roles (Step 6). (c) **Leaked secrets** — solved by keyless auth. (d) **Weak labels** — vague or chunk-dependent evidence that silently breaks later measurement.

**Q6. How would a production foundations stage differ from this notebook?**
It would use a **managed identity** instead of a human login, export telemetry to a persistent collector (not in-memory), store the evaluation set and corpus manifest in version control or a dataset registry, and automate corpus ingestion as a pipeline rather than a notebook run. Notebooks 11 and 12 take these steps; this notebook establishes the patterns in a reviewable, single-machine form first.

**Q7. Why does the setup verify a checksum instead of trusting the upload's "success" message?**
A successful upload confirms bytes were sent, not that the right bytes arrived intact. A checksum (SHA-256) is a content fingerprint: re-computing it locally and comparing to the manifest proves the stored file is byte-for-byte what you intended. This is the difference between "the code ran" and "the data is correct" — a distinction this whole course insists on.

**Q8. How does this stage connect to the next one?**
Everything here is saved to `.artifacts/setup-handoff.json` (the verified corpus location and manifest) and `.artifacts/eval-questions.json` (the labeled questions). Notebook 01 (Ingestion & Parsing) reads those files — not this kernel's variables — and begins turning the raw play texts into clean, structured, source-linked blocks. The handoff pattern you see here repeats at every stage boundary for the rest of the series.
